# 11.02 集合交算子与带宽实验

## 小节概述

本节把 11.01 的集合交语义映射为两个输入、一个输出的 Ascend C 逐元素算子。DenseAnd 和 BitmapAnd 使用同一多核/Tile/队列框架，只改变物理 dtype 与任务总量；这样性能差异主要来自表示压缩和 GM 流量，而不是不等价的实现。

### 学习前置要求

- 已完成 [11.01 位图集合的数据结构设计](./11.01_bitmap_set_structure.ipynb)；
- 能够计算有效字数、物理行跨度和集合交的 CPU Golden；
- 已进入带有 CANN 与昇腾 NPU 的 CANNLab 环境。

### 本节目标

完成 11.02 后，你应能够：

1. 为 DenseAnd 与 BitmapAnd 推导连续多核切分和 UB Tile；
2. 说明三条单缓冲队列的空间占用与同步关系；
3. 使用正确性、输出 Guard 和非法参数用例验证算子边界；
4. 区分物理流量压缩比与设备实测加速比。

<strong>建议用时：</strong> 60 分钟。<strong>本节产出：</strong> Dense/Bitmap 边界运行记录、物理流量推导和一组 NPU 计时结果。

## 教程内容

In [ ]:
from pathlib import Path
import json
import re
import shutil
import subprocess

def locate_chapter():
    relative = Path('contrib/tutorials/data_structures_compute/11_bitmap_set_bandwidth')
    for candidate in [Path.cwd(), Path.cwd() / relative, *[p / relative for p in Path.cwd().parents]]:
        if (candidate / 'src' / 'demo' / 'bitmap_and.asc').is_file():
            return candidate.resolve()
    raise FileNotFoundError('未找到实验 11 目录。')

CHAPTER_DIR = locate_chapter()
SOURCE = CHAPTER_DIR / 'src' / 'demo' / 'bitmap_and.asc'
CMAKE_FILE = CHAPTER_DIR / 'src' / 'demo' / 'CMakeLists.txt'
WORK = Path('/tmp/cannlab_data_structures_compute/11_bitmap_set_bandwidth/11.02_bitmap_and_operator')
print('source:', SOURCE)
print('npu   :', 'PASS' if subprocess.run(['npu-smi', 'info'], capture_output=True).returncode == 0 else 'FAILED')

### 1. 算子规格与公平比较

设批量大小为 <code>N</code>、全集大小为 <code>U</code>：

$$C[n,i]=A[n,i]\land B[n,i].$$

| Kernel | 物理 dtype | 每行有效单位 | 每行物理单位 |
| --- | --- | ---: | ---: |
| DenseAnd | <code>uint8</code> | <code>U</code> 个元素 | <code>AlignUp(U, 32)</code> 个元素 |
| BitmapAnd | <code>uint32</code> | <code>CeilDiv(U, 32)</code> 个字 | <code>AlignUp(validWords, 8)</code> 个字 |

两个输入在计时前完成生成和 H2D，D2H 与 CPU Golden 也在计时外；计时间隔只包含多次 Kernel 发射和一次流同步。

### 2. 数据通路与同步

![DenseAnd 与 BitmapAnd 数据通路](images/bitmap_dataflow.svg)

Kernel 采用 <code>Init → Process → CopyIn/Compute/CopyOut</code>。<code>EnQue/DeQue</code> 建立 MTE 与 Vector 阶段的同步；每条 <code>TQue&lt;..., 1&gt;</code> 使用 <code>InitBuffer(..., 1, ...)</code> 的一个物理槽位。模板参数深度与物理 Buffer 数是两个概念，本实验都明确设为 1。CANN 9.0 在 Atlas A2/A3 上正式支持 <code>uint16</code> 的基础 AND，因此 Dense 的 <code>uint8</code> 与 Bitmap 的 <code>uint32</code> LocalTensor 都通过 <code>ReinterpretCast&lt;uint16_t&gt;</code> 执行；重解释只改变元素视图，不改变任何 bit、GM 布局或物理流量。

### 3. 多核与 UB 切分

先把批量物理行展平为 <code>totalUnits=N*physicalUnitsPerRow</code>。Host 以约 4 KB/核估算目标核数，再把普通 Block 向上对齐到 512 个物理单位；最后一个 Block 使用真实 <code>blockTail</code>。

每条队列分配 8192 Byte，三条单缓冲队列共用 <code>3*8192=24576</code> Byte。Dense 的 <code>tileUnits=8192</code>，Bitmap 的 <code>tileUnits=2048</code>。所有行、Block 与 Tile 均保持 32 Byte 对齐，因此可使用对齐版 <code>DataCopy</code>；如果去掉物理行填充，就必须改用带扩展参数的 <code>DataCopyPad</code> 处理非对齐尾部。

这里的 8 KB Tile 和单缓冲是为了固定执行框架、隔离“数据表示”这一变量，并不代表生产算子的峰值配置。通用性能建议通常希望单次搬运达到 16 KB 或继续探索双缓冲；这些属于后续流水调优，不纳入本章结论。

In [ ]:
def ceil_div(value, divisor):
    return (value + divisor - 1) // divisor

def align_up(value, alignment):
    return ceil_div(value, alignment) * alignment

def make_plan(batch, universe, element_bytes):
    valid = universe if element_bytes == 1 else ceil_div(universe, 32)
    physical = align_up(valid, 32 // element_bytes)
    total = batch * physical
    min_units_per_core = ceil_div(4096, element_bytes)
    target_cores = min(20, max(1, ceil_div(total, min_units_per_core)))
    block_former = align_up(ceil_div(total, target_cores), 512)
    block_num = ceil_div(total, block_former)
    block_tail = total - (block_num - 1) * block_former
    tile_units = 8192 // element_bytes
    return dict(valid=valid, physical=physical, total=total, block_num=block_num,
                block_former=block_former, block_tail=block_tail, tile_units=tile_units)

for name, size in [('dense', 1), ('bitmap', 4)]:
    print(name, make_plan(256, 4096, size))

### 4. 对照真实源码

下面只抽取关键行。阅读完整文件时重点确认：核间偏移只加一次；普通/尾 Block 和普通/尾 Tile 分别计算；三条队列均计入空间；GM 上没有逐元素 <code>GetValue/SetValue</code>。

In [ ]:
source_lines = SOURCE.read_text(encoding='utf-8').splitlines()
needles = ['StudentBlockOffset', 'blockOffset =', 'blockUnits_ =', 'InitBuffer(',
           'DataCopy(leftLocal', 'ComputeSetAnd(', 'ReinterpretCast<uint16_t>', 'DataCopy(outputGm_']
for number, line in enumerate(source_lines, start=1):
    if any(needle in line for needle in needles):
        print(f'{number:>4}: {line}')

### 5. 干净构建

构建目录固定在 <code>/tmp/cannlab_data_structures_compute/11_bitmap_set_bandwidth/11.02_bitmap_and_operator</code> 下，不修改课程源码。<strong>检查点：</strong> 配置与构建命令的返回码都应为 0，并生成 <code>bitmap_and</code>。

In [ ]:
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
commands = [
    ['cmake', '-S', str(CMAKE_FILE.parent), '-B', str(WORK), '-DNPU_ARCH=dav-2201'],
    ['cmake', '--build', str(WORK), '-j2'],
]
for command in commands:
    print('$', subprocess.list2cmdline(command))
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    result.check_returncode()
EXECUTABLE = next(path for path in WORK.rglob('bitmap_and') if path.is_file())
print('executable:', EXECUTABLE)

### 6. 正确性与边界验证

先运行 <code>U=31/32/33</code>，再运行一个大规模非对齐位图。结果必须同时满足 <code>correctness=PASS</code> 与 <code>guard=PASS</code>；整数位运算不使用 <code>rtol/atol</code>。

In [ ]:
cases = [('dense', 1, 31), ('dense', 2, 32), ('dense', 3, 33),
         ('bitmap', 1, 31), ('bitmap', 2, 32), ('bitmap', 3, 33),
         ('bitmap', 32, 1_000_003)]
for layout, batch, universe in cases:
    command = [str(EXECUTABLE), '--layout', layout, '--batch', str(batch),
               '--universe', str(universe), '--warmup', '1', '--iterations', '1']
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout.strip())
    assert result.returncode == 0 and 'correctness=PASS' in result.stdout and 'guard=PASS' in result.stdout
print('NPU_BOUNDARY_CHECK PASS')

### 7. 带宽实验：不要把压缩比写成加速比

两个输入和一个输出使物理流量为：

```text
denseBytes  = 3 * N * AlignUp(U, 32)
bitmapBytes = 3 * N * AlignUp(CeilDiv(U, 32), 8) * 4
```

输出同时给出 <code>physical_gb_s</code> 与 <code>logical_gb_s</code>。前者描述实际物理字节，后者描述每秒完成多少稠密成员语义。理论压缩比、物理带宽和 Kernel 加速比是三个不同指标。

In [ ]:
metric_pattern = re.compile(r'([a-z_]+)=([^\s]+)')
records = {}
for layout in ['dense', 'bitmap']:
    command = [str(EXECUTABLE), '--layout', layout, '--batch', '256', '--universe', '4096',
               '--warmup', '10', '--iterations', '100', '--seed', '7']
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=True)
    print(result.stdout.strip())
    metric_line = [line for line in result.stdout.splitlines() if line.startswith('METRIC ')][-1]
    records[layout] = dict(metric_pattern.findall(metric_line))
dense_us = float(records['dense']['avg_kernel_us'])
bitmap_us = float(records['bitmap']['avg_kernel_us'])
print('measured_speedup =', dense_us / bitmap_us)
print('physical_compression =', int(records['dense']['physical_bytes']) / int(records['bitmap']['physical_bytes']))

<strong>结果解释：</strong> 请保留本机输出，不预填固定倍数。大 <code>U</code> 时 Bitmap 的物理流量应接近 Dense 的 1/8，但实测加速会受启动开销、核数、Tile 数和设备负载影响；短行还会受到每行 32 Byte 填充影响。若加速与假设不符，先检查计时边界和物理字节数，不能删除不符合预期的数据。

### 8. 非法参数必须在发射前拒绝

以下用例应返回 2，且不能输出 <code>METRIC</code>。

In [ ]:
invalid = [
    ['--layout', 'dense', '--batch', '0', '--universe', '32'],
    ['--layout', 'bitmap', '--batch', '1', '--universe', '0'],
    ['--layout', 'unknown'],
    ['--layout', 'bitmap', '--iterations', '0'],
]
for args in invalid:
    result = subprocess.run([str(EXECUTABLE), *args], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print('returncode=', result.returncode, result.stdout.splitlines()[0])
    assert result.returncode == 2 and 'METRIC ' not in result.stdout
print('INVALID_ARGUMENT_CHECK PASS')

## 课后练习

1. 分别推导 <code>N=3,U=33</code> 的 Dense/Bitmap 物理流量。
2. 说明为什么本实验选择单缓冲而不是把双缓冲作为变量。
3. 增加 <code>N=32,U=1,048,576</code> 和 <code>N=32,U=1,000,003</code>，比较对齐与非对齐 Shape。
4. 若取消每行填充，说明 CopyIn/CopyOut 为什么必须改用 <code>DataCopyPad</code>。

In [ ]:
SHOW_ANSWER = False
answer_file = CHAPTER_DIR / 'answer' / '11.02_bitmap_and_operator' / 'answers.md'
print(answer_file.read_text(encoding='utf-8') if SHOW_ANSWER else '将 SHOW_ANSWER 改为 True 后重新运行。')

## 本节小结与完成检查

完成本节时应能说明：Dense 与 Bitmap 为什么保持同一执行框架；普通/尾 Block 和普通/尾 Tile如何覆盖完整物理数组；三队列为何只占 24576 Byte；为什么压缩比不能替代实测加速比。

继续完成 [11.03 独立实践与验收](./11.03_chapter_test.ipynb)。